In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone 4

In [2]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")


label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}


train["answer_encoded"] = train["answer"].map(label_map)


print("Original answer:", train.loc[150, "answer"])
print("Encoded label:", train.loc[150, "answer_encoded"])

Original answer: C
Encoded label: 2


In [3]:
formatted_input = str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, "B"])

print(formatted_input)

print("Character length:", len(formatted_input))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Character length: 407


In [6]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")


label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

idx = 0

prompt = str(train.loc[idx, "prompt"])
choices = [
    str(train.loc[idx, "A"]),
    str(train.loc[idx, "B"]),
    str(train.loc[idx, "C"]),
    str(train.loc[idx, "D"]),
    str(train.loc[idx, "E"]),
]

encoding = tokenizer(
    [prompt] * 5,
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)


labels = torch.tensor([label_map[train.loc[idx, "answer"]]])

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)

print("Loss:", outputs.loss)
print("Loss shape:", outputs.loss.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loss: tensor(1.6032, grad_fn=<NllLossBackward0>)
Loss shape: torch.Size([])


In [9]:
import torch
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, TaskType, get_peft_model


model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)
print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")

model.print_trainable_parameters()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco

Trainable parameters: 295681
Total parameters: 109778690
Trainable %: 0.2693%
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


In [8]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.2 MB/s eta 0:00:00:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [10]:
import torch

model.eval()


row = train.iloc[0]

prompt = str(row["prompt"])
choices = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"]),
]

encoding = tokenizer(
    [prompt] * 5,
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,  
    return_tensors="pt",
)

with torch.no_grad():
    outputs = model(
        input_ids=encoding["input_ids"].unsqueeze(0),
        attention_mask=encoding["attention_mask"].unsqueeze(0),
    )

logits = outputs.logits
probs = torch.softmax(logits, dim=-1)

print("Logits:", logits)
print("Probabilities:", probs)
print("Probability of Option E:", probs[0, 4].item())
print("Rounded:", round(probs[0, 4].item(), 4))

Logits: tensor([[-0.4204, -0.3582, -0.4045, -0.4815, -0.5234]])
Probabilities: tensor([[0.2031, 0.2162, 0.2064, 0.1911, 0.1832]])
Probability of Option E: 0.18324312567710876
Rounded: 0.1832
